# STEP 4-A — 2단계 베이스라인

## 왜 한 번에 7클래스가 아니라 2단계인가

이 기능의 목적은 **"보호자에게 의심된다까지 알려주기"** 입니다.
그러면 가장 중요한 판단은 **"병원에 가봐야 하나?"** 이고, 그건 이진 문제입니다.
병변 종류를 6개로 나누는 건 그 다음 이야기죠.

A7(정상)까지 한 번에 7클래스로 풀면 두 문제가 섞입니다:

- A7(정상)을 A2(비듬)로 틀리는 것과 A2를 A3로 틀리는 것은
  **임상적 무게가 완전히 다른데**, 7클래스 손실함수는 둘을 똑같이 취급합니다.
- "놓치지 않는 게 우선" 이라는 **재현율 우선 임계값을 1단계에만** 걸 수가 없습니다.

그래서 나눕니다:

| 단계 | 하는 일 | 데이터 (VL01) | 기준 |
|---|---|---|---|
| **1단계** | 정상(A7) vs 이상 | 22,815 : 23,070 — 거의 5:5 | **재현율 ≥ 0.95** |
| **2단계** | 병변 6종 (A1~A6) | 23,070장, 6.4배 불균형 | macro-F1 |

## 순서대로 실행하시면 됩니다

| | 하는 일 | 학습 | 대략 시간 |
|---|---|---|---|
| 1 | 크롭 검증 — 배율 지름길·`full` 천장·하한선 | — | 2분 |
| 2 | 2단계 뷰 (1단계 크롭 자동 결정) | — | 1분 |
| 3 | 1단계 학습 + 게이트 | ✅ | ~85분 |
| 4 | 2단계 학습 + 게이트 | ✅ | ~75분 |
| 5 | 두 단계 이어붙인 실제 성능 | — | 3분 |
| 6 | 실사용 견고성 (배율·위치 교란) | — | 8분 |
| 7 | 결과 저장 | — | 1분 |

**위에서 아래로 전부 실행하면 됩니다.** Kaggle 은 `Save & Run All (Commit)`.
전체 약 3시간이고, 세션이 끊겨도 **그냥 다시 돌리면** 끝난 학습은 건너뛰고
끊긴 학습은 그 에폭부터 이어갑니다.

> 입력 해상도는 **384** 입니다. 224 와 비교해 배율 하락이 1단계 17.2% → 9.1%,
> 2단계 29.5% → 20.4% 로 줄어 채택했습니다. 그 비교 실험 자체는 결론이 났으므로
> 노트북에서 뺐고, 기록은 [`docs/results/`](../docs/results/) 에 있습니다.
> 남은 2단계 20.4% 는 `03b_증강_배율강건성.ipynb` 에서 다룹니다.

> ⚠️ 1단계와 2단계는 **같은 개체 단위 분할**을 씁니다.
> 단계별로 따로 나누면 1단계 검증에 쓴 강아지가 2단계 학습에 들어가 누수가 생깁니다.
> `stages.to_stage1/to_stage2` 가 `fold`/`is_holdout`/`group` 을 그대로 물려받습니다.

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
NAME   = "deeplearning_test"
# ⚠️ 브랜치를 "main" 으로 **못 박으면 안 됩니다.** 아래 reset --hard 가
#    작업 브랜치를 통째로 덮어써서, 방금 만든 코드가 사라진 채로 몇 시간을
#    돌게 됩니다. 이미 리포 안에서 돌고 있으면 **지금 브랜치를 그대로 씁니다.**
#    바꾸려면 환경변수:  export DOG_SKIN_BRANCH=main
# ★ 이 노트북이 사는 브랜치. **여기서 못 박지 않으면 "main" 을 받습니다.**
#    캐글/콜랩은 리포가 없는 상태로 시작해서 아래 _ROOT 탐색이 실패하고,
#    예전 기본값이 "main" 이었습니다. main 이 뒤처져 있으면 **셀은 최신인데
#    src/ 만 옛것**인 채로 돕니다 — 실제로 며칠 그랬습니다 (main 75445c0).
#    첫 셀은 그 상태에서도 "코드 버전 …" 을 태연히 찍습니다.
NB_BRANCH = "claude/dog-disease-diagnosis-model-1s6jtf"
BRANCH = os.environ.get("DOG_SKIN_BRANCH", "")
_cwd   = os.getcwd()

# ⚠️ "지금 리포 안인가" 를 **폴더 이름으로만** 보면 안 됩니다. 주피터에서
#    notebooks/*.ipynb 를 열면 cwd 가 `.../deeplearning_test/notebooks` 라
#    이름이 안 맞고, 그러면 **리포 안에 리포를 또 clone** 합니다
#    (실제로 런팟에서 .../notebooks/deeplearning_test 가 생겼습니다).
#    위로 거슬러 올라가며 **진짜 리포 루트**를 찾습니다.
_p = os.path.abspath(_cwd)
_ROOT = None
while True:
    if (os.path.isdir(os.path.join(_p, ".git"))
            and os.path.isfile(os.path.join(_p, "src", "env.py"))):
        _ROOT = _p
        break
    _up = os.path.dirname(_p)
    if _up == _p:
        break
    _p = _up

if _ROOT:
    DIR = _ROOT           # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
    if not BRANCH:
        BRANCH = subprocess.run(["git", "-C", DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip() or "main"
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

BRANCH = BRANCH or NB_BRANCH

# ⚠️ 예전엔 fetch/reset 을 **둘 다 check=False** 로 불렀습니다. 실패해도 조용히
#    넘어가서, 캐글 클론이 **지워진 커밋(75445c0)에 붙박인 채 며칠을 돌았습니다.**
#    src/ 를 아무리 고쳐 푸시해도 안 실렸고, 첫 셀은 "코드 버전 …" 을 태연히
#    찍었습니다. 그 줄을 믿을 수 없다는 게 제일 나빴습니다.
#    → 이제 실패하면 **말하고, 클론을 지우고 다시 받습니다.**
#    (Kaggle Persistence 를 'Files' 로 켜두면 /kaggle/working 이 살아남아
#     낡은 클론이 계속 재사용됩니다 — 그 경우에도 여기서 복구됩니다.)
def _git(*args, cwd=None):
    return subprocess.run(["git", *args], capture_output=True, text=True, cwd=cwd)


def _fresh_clone(dst, branch):
    import shutil as _sh
    _sh.rmtree(dst, ignore_errors=True)
    r = _git("clone", "-b", branch, "--depth", "1", REPO, dst)
    if r.returncode != 0:
        raise RuntimeError("git clone 실패:\n" + (r.stderr or "")[-800:])


_need_clone = not os.path.isdir(os.path.join(DIR, ".git"))
if not _need_clone:
    # shallow clone 이라 origin/<브랜치> 대신 FETCH_HEAD 로 맞춥니다
    # (히스토리가 갈리면 origin/<브랜치> 가 옛 커밋을 가리킨 채 남습니다)
    r = _git("-C", DIR, "fetch", "--depth", "1", "origin", BRANCH)
    if r.returncode != 0:
        print("⚠️ git fetch 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
        _need_clone = True
    else:
        r = _git("-C", DIR, "reset", "--hard", "FETCH_HEAD")
        if r.returncode != 0:
            print("⚠️ git reset 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
            _need_clone = True

if _need_clone:
    _fresh_clone(DIR, BRANCH)

# ★ 정말 최신인지 **확인**합니다. 위가 다 성공해도 여기서 한 번 더 봅니다 —
#   "최신이라고 믿었는데 아니었다" 가 이 프로젝트에서 가장 비쌌던 실패입니다.
_local = _git("-C", DIR, "rev-parse", "HEAD").stdout.strip()
_remote = _git("-C", DIR, "ls-remote", REPO, f"refs/heads/{BRANCH}").stdout.split()
_remote = _remote[0] if _remote else ""
if _remote and _local and not _remote.startswith(_local[:8]) and not _local.startswith(_remote[:8]):
    print("\n" + "!" * 66)
    print(f"🚨 코드가 최신이 아닙니다 — 로컬 {_local[:8]} / 원격 {_remote[:8]}")
    print("   클론을 지우고 다시 받습니다.")
    print("!" * 66 + "\n")
    _fresh_clone(DIR, BRANCH)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", _git("-C", DIR, "log", "--oneline", "-1").stdout.strip())
print("브랜치      :", BRANCH,
      f"(원격 {_remote[:8]})" if _remote else "(원격 확인 실패)")
if BRANCH != NB_BRANCH:
    print(f"⚠️ 이 노트북이 만들어진 브랜치({NB_BRANCH})가 아닙니다 —")
    print("   src/ 가 셀보다 뒤처져 있을 수 있습니다. 아래 [nb] 줄을 꼭 보세요.")

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

# ⚠️ 임대 GPU 이미지(런팟 등)의 파이썬은 **externally managed** 입니다 (PEP 668).
#    그냥 설치하면 첫 시도가 통째로 거부돼서, 재시도 로직이 있어도 무서운
#    에러 덩어리가 먼저 찍힙니다. 처음부터 허용해두면 그 소음이 없습니다.
#    Colab/Kaggle 에는 이 제약이 없어서 이 변수는 무해합니다.
os.environ["PIP_BREAK_SYSTEM_PACKAGES"] = "1"
os.environ["UV_BREAK_SYSTEM_PACKAGES"] = "1"

# ⚠️ Colab/Kaggle 에는 numpy·pandas·sklearn 이 이미 있지만 **임대 GPU 이미지엔
#    torch 만 있는 경우가 많습니다** (런팟에서 `No module named 'pandas'` 로
#    막혔습니다). 그렇다고 매번 다 깔면 Colab 에서 버전이 흔들리므로
#    **없는 것만** 깝니다.
_NEED = {                       # import 이름 → pip 이름
    "numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "PIL": "Pillow",
    "sklearn": "scikit-learn", "cv2": "opencv-python-headless", "tqdm": "tqdm",
    "matplotlib": "matplotlib", "timm": "timm", "imagehash": "imagehash",
    "pytorch_grad_cam": "grad-cam", "albumentations": "albumentations",
}
import importlib.util as _ilu

_PKGS = [pip for mod, pip in _NEED.items() if _ilu.find_spec(mod) is None]
if _PKGS:
    print(f"[env] 없는 패키지 {len(_PKGS)}개를 깝니다: {_PKGS}")
else:
    print("[env] 필요한 패키지가 전부 있습니다 — 설치를 건너뜁니다")

# ⚠️ 일부 이미지(런팟 PyTorch 등)는 파이썬이 **externally managed** 라
#    (PEP 668) --system 설치를 거부합니다. Colab/Kaggle 에는 없는 문제라
#    처음엔 안 넣었다가 런팟에서 첫 셀이 바로 죽었습니다.
#    --break-system-packages 를 붙여 한 번 더 시도합니다.
def _install(args: list[str]) -> bool:
    return subprocess.run(args, check=False).returncode == 0


_ok = not _PKGS          # 깔 게 없으면 이미 성공입니다
if _PKGS and _install([sys.executable, "-m", "pip", "install", "-q", "uv"]):
    _base = [sys.executable, "-m", "uv", "pip", "install", "-q", "--system"]
    _ok = _install(_base + _PKGS)
    if not _ok:
        _ok = _install(_base + ["--break-system-packages"] + _PKGS)
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    _p = [sys.executable, "-m", "pip", "install", "-q"]
    if not _install(_p + _PKGS):
        _install(_p + ["--break-system-packages"] + _PKGS)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-09-04.5"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


## 0-b. 로컬에서 만든 데이터 불러오기 + 중단 대비

한국 PC 에서 `prepare_local.py` 로 전처리한 `dogskin_prepared.zip` 을 가져옵니다.

> 🚨 **AI Hub 는 해외 IP 다운로드를 차단**해서 Colab 에서는 원본을 받을 수 없습니다.
> 다운로드·전처리는 로컬에서, 학습만 여기서 합니다.
> → [`docs/cautions/06`](../docs/cautions/06_해외IP_다운로드_차단_우회.md)

**준비**
- **Colab** : `dogskin_prepared.zip` 을 Google Drive 에 올려두세요
- **Kaggle** : 같은 zip 을 Dataset 으로 올리고 우측 **Add Input** 으로 붙이세요.
  Kaggle 이 zip 을 알아서 풀어두므로 `crops/` `manifests/` 가 바로 보입니다.
  → 자세한 절차: [`docs/cautions/09`](../docs/cautions/09_세션이_끊겼을_때.md)

### ⚠️ Colab 세션은 예고 없이 끊깁니다

3번+4번 학습이 합쳐서 1시간 반쯤 걸립니다. 그 사이에 세션이 끊기면
**`/content` 가 통째로 사라집니다** — 크롭 이미지도, 체크포인트도요.

그래서 체크포인트는 매 에폭 **Drive 로 복사**해 둡니다.
끊기면 이 노트북을 **위에서부터 그냥 다시 돌리세요.** 끝난 학습은 건너뛰고,
끊긴 학습은 그 다음 에폭부터 이어갑니다 (옵티마이저 상태까지 복원).

| 상황 | 다시 돌렸을 때 |
|---|---|
| 3번은 끝, 4번 도중에 끊김 | 3번 `⏭️ 건너뜁니다` → 4번 `▶️ epoch N 부터 이어서` |
| 4번을 20에폭에서 끊김 | 21에폭부터 (앞의 20에폭 다시 안 함) |
| 다 끝났는데 또 돌림 | 둘 다 건너뜀, 평가만 다시 |
| 에폭을 더 늘리고 싶다 | `epochs` 만 키우면 이어서 연장합니다 |
| 처음부터 다시 하고 싶다 | `train.fit(..., resume=False)` |

> 💡 **Drive 마운트를 건너뛰면 이 보호가 없습니다.** 아래 셀이 경고합니다.
> 체크포인트는 실험당 약 600MB 를 씁니다 (`best.pt` 200MB + `last.pt` 400MB).
> Drive 용량이 빡빡하면 Drive 의 `MyDrive/dogskin_work/checkpoints/*/last.pt` 를 지우세요
> (`best.pt` 만 남기면 이어받기는 못 하지만 평가는 됩니다).


In [ ]:
# Drive 마운트는 **진짜 Colab VM** 에서만 시도합니다.
# ⚠️ Kaggle 에도 google.colab 패키지와 /content 가 있어서, 환경 판정을 잘못하면
#    Kaggle 에서 drive.mount() 를 부르고 NotImplementedError 로 죽습니다.
if env.can_mount_drive():
    env.mount_drive()
else:
    print(f"[env] {E.env} — Drive 마운트 없이 진행합니다")

# 전처리 결과를 붙입니다. 두 가지 형태를 다 받습니다:
#   · Colab  : Drive 의 dogskin_prepared.zip → 로컬 디스크로 해제
#   · Kaggle : /kaggle/input/<데이터셋>/crops,manifests → 링크만 연결
#              (Kaggle 은 업로드한 zip 을 알아서 풀어둡니다. 복사하면 20GB 제한에 걸려요)
env.load_prepared()          # 경로를 직접 주려면: env.load_prepared("/kaggle/input/dogskin-prepared")

# ⚠️ train 은 아래 셀에서 import 하지만 여기서 먼저 씁니다 — 여기서 불러둡니다.
#    (실제로 NameError 로 5분 돌다 죽었습니다. 셀 순서를 믿지 말 것)
from src import train

# ── 예전 실행의 release 를 붙였으면 가져옵니다 (선택) ──────────────────
#    ★ **안 바꾼 단계를 다시 학습하지 않기 위해서입니다.**
#
#    예: 1단계 크롭만 f320 으로 바꾸는 실행에서, 2단계(m2.5·resnet50)는
#    설정이 그대로입니다. 그런데 다시 학습하면 25에폭(~1.5시간)이 날아가고,
#    더 나쁘게는 가중치가 미묘하게 달라져서 **파이프라인 숫자 변화에 "바꾼
#    단계의 효과" 와 "안 바꾼 단계의 잡음" 이 섞입니다.**
#
#    붙일 게 없으면 조용히 넘어갑니다 — 처음 돌리는 경우가 그렇습니다.
_prev = train.import_previous_run(verbose=True)
if _prev and _prev.get("checkpoints"):
    print(f"\n[인계] 가져온 실험: {_prev['checkpoints']}")
    print("   설정이 그대로인 단계는 아래 학습 셀에서 ⏭️ 로 건너뜁니다.")
    print("   ⚠️ 다시 학습하고 싶으면 train.fit(..., resume=False) 로 부르세요.")
else:
    print("\n[인계] 붙어 있는 이전 실행이 없습니다 — 두 단계 다 처음부터 학습합니다.")

# 세션이 끊겨도 남는 저장소 확인
_persist = env.persist_root()
if _persist is None:
    print("\n🚨 세션 밖 저장소가 없습니다 — 지금 학습하면 끊길 때 체크포인트가 사라집니다.")
    print("   위 셀에서 Drive 마운트가 됐는지 확인하세요 (env.mount_drive()).")
else:
    print(f"\n✅ 중단 대비 저장소: {_persist}")
    if E.env == "kaggle":
        print("   ⚠️ Kaggle 은 세션이 끝나면 /kaggle/working 이 사라질 수 있습니다.")
        print("      · 짧게 확인만 할 때  : 그냥 진행 (세션 안에서는 이어받기가 됩니다)")
        print("      · 긴 학습을 돌릴 때  : 우측 상단 [Save Version] →")
        print("                             'Save & Run All (Commit)' 로 돌리세요.")
        print("                             브라우저를 닫아도 끝까지 돌고, 출력이 보존됩니다.")
        print("      · 설정에 Persistence 항목이 보이면 'Files' 로 켜두면 더 안전합니다")
    else:
        print("   매 에폭 체크포인트를 여기로 복사합니다. 세션이 끊기면 노트북을 처음부터")
        print("   다시 돌리세요 — 끝난 학습은 건너뛰고 끊긴 학습만 이어서 합니다.")

In [ ]:
import torch
from src import labels, split, crop, data, models, train, evaluate, stages
from src.config import CLASSES_STAGE1, NORMAL_LABEL

# ★ GPU 없이 진행하면 20~30배 느립니다. 없으면 여기서 멈춥니다.
#   (Colab 무료 한도를 넘기면 말없이 CPU 런타임을 줍니다 — 이걸 막습니다)
env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"

df = labels.load(env.work_root()/"manifests"/"manifest_final.parquet")
print(f"{len(df):,}행 / 개체 {df['animal_id'].nunique():,}마리")
print("클래스 분포:", df["label"].value_counts().to_dict())
print("사용 가능한 크롭 태그:", crop.available_tags())

# 로컬에서 개체 단위 분할까지 끝냈으므로 fold/holdout 컬럼이 들어 있습니다
split.verify(df, fold=0, strict=True)

## 1. 크롭 검증 — 숫자로 먼저, 눈으로는 딱 하나만

여기서 확인해야 하는 건 "병변이 맞는가" 가 **아닙니다.** 그건 수의사의 일이고,
저 데이터의 라벨은 이미 수의사가 붙인 것입니다. 우리가 확인할 건 다른 겁니다:

> **크롭이 라벨을 다른 경로로 흘리고 있지 않은가.**

무슨 뜻이냐면 — A7(정상)에도 라벨러가 "촬영한 피부 부위" 박스를 찍어놨습니다.
그래서 A7 도 `area_ratio` 값이 있습니다. 그 자체는 정상인데, 만약
**정상 박스가 병변 박스보다 계통적으로 크다면** 크롭의 확대 배율만 봐도
정상/병변이 티가 납니다. 그러면 모델은 피부를 안 보고 **줌 레벨을 셉니다.**

이런 지름길(shortcut)은 검증 점수를 **높게** 만듭니다. 그래서 숫자로 잡아야 합니다.
`crop.audit()` 이 그걸 포함해 5가지를 재줍니다 — 의학 지식이 필요 없습니다.

In [ ]:
# ★ m2.5 확정 (2026-08-22, STEP 4C). 03c 에서 같은 실행으로 비교한 결과
#   배율 하락 23.2% → 18.8%, A1 recall 0.560 → 0.622.
#   근거: docs/results/STEP4C_크롭비교_실측.md
#
#   ⚠️ **1단계는 이 값을 따라오지 않습니다.** 아래 18번 셀의
#   crop.choose_stage1_tag() 가 따로 고릅니다 (우선순위 f320 > full > ROI).
#   ROI 크롭은 창 크기가 정답을 흘려서 1단계에 못 씁니다.
BEST_CROP = "m2.5"                       # 2단계 크롭. 병변 형태를 보려면 ROI 가 필요합니다
# ★ 384 채택 (2026-08-21 해상도 실험). 감사 [2] 와 맞춰야 의미가 있습니다.
#   224 → 384 로 올려 배율 하락이 1단계 17.2%→9.1%, 2단계 29.5%→20.4% 로 줄었습니다.
#   디스크 크롭이 512px 라 재크롭 없이 가능하고, 배치는 자동으로 줄어듭니다(T4 → 12).
#   근거: docs/results/STEP4A_베이스라인_실측.md
IMG_SIZE  = 384

# ★ STEP 6 (03d) 에서 1단계 설정을 갈아끼웠습니다 — 2×2 실측 결과입니다.
#   근거: docs/results/STEP6_1단계_2x2_실측.md
#
#     resnet50   / default      AUROC 0.8116   흐림 하락 55.4%   ← 이전
#     effnetv2_s / photometric  AUROC 0.8284   흐림 하락 14.0%   ← 채택
#
#   ⚠️ val AUROC 최고는 effnetv2_s/default(0.8359) 였지만 **안 골랐습니다.**
#      흐림 하락 47.6% 로 화질 지름길에 기대고 있고, AUROC 차이 0.0075 는
#      잡음(±0.01) 안입니다. val 점수로 고르다가 holdout 에서 무너진 게
#      STEP 5 의 실패였습니다 (0.8143 → 0.7412).
STAGE1_MODEL = "effnetv2_s"
STAGE1_AUG   = "photometric"
# 03d(서브셋 55%·12에폭)에서 best epoch 이 **5**, 조기 종료가 10에폭이었습니다.
# 풀 데이터는 1.8배라 best 가 뒤로 밀리겠지만 5배까지는 아닙니다.
# patience=5 라 18이면 여유가 있고, 더 필요하면 마지막 에폭이 best 로 찍혀 알려줍니다.
STAGE1_EPOCHS = 18

# 첫 실측 기준선 (conservative bb×0.1 / 1단계 8ep / 2단계 12ep / 배치 16).
# 다시 돌릴 필요 없습니다 — 아래 게이트들이 이 값과 비교해 개선폭을 보여줍니다.
# 근거: docs/results/STEP4A_베이스라인_실측.md
BASELINE = {"stage1_auroc": 0.8031, "stage2_macro_f1": 0.4865, "stage2_a6_recall": 0.328}

# 실측 참고값. 이번 실행이 여기서 크게 벗어나면 데이터·환경을 먼저 의심하세요.
# ⚠️ 1단계 기준값은 **f320 서브셋(STEP 9-A)** 입니다. 풀 데이터라 조금 더 오를 수 있습니다.
MEASURED_384 = {"stage1_auroc": 0.9477,      # f320 · 서브셋 12에폭 (2026-08-23, 03e)
                "stage1_blur_drop": 0.031,   # f320 — full 은 11.1% 였습니다
                "stage2_macro_f1": 0.5395,   # m2.5 (2026-08-22, 03c)
                "stage2_scale_drop": 0.188}  # m2.5

# ── 붙어 있는 크롭 확인 — 학습 전에 여기서 걸러야 합니다 ─────────────
_tags = crop.available_tags()
print(f"붙어 있는 크롭 태그: {_tags}")
if BEST_CROP not in _tags:
    raise SystemExit(
        f"❌ 2단계 크롭 '{BEST_CROP}' 이 없습니다. 붙어 있는 것: {_tags}\n"
        f"   Kaggle 이면 [Add Input] 으로 그 크롭 데이터셋을 붙이세요.")
# ★ STEP 9-A (2026-08-23): 1단계 크롭을 f320 으로 확정했습니다.
#   val AUROC 0.8272(full) → 0.9477(f320), 흐림 하락 11.1% → 3.1%.
#   근거: docs/results/STEP9A_1단계_f320_실측.md
#   choose_stage1_tag() 가 f320 을 full 보다 먼저 고르므로, 붙여만 두면 됩니다.
if not any(crop.fixed_of_tag(t) for t in _tags):
    print("\n🚨 f320 (고정 픽셀) 크롭이 안 붙어 있습니다.")
    print("   1단계는 f320 으로 확정됐습니다 (STEP 9-A: AUROC +0.12, 흐림 하락 −8%p).")
    if "full" in _tags:
        print("   ⚠️ full 로 떨어져서 **멈추지 않고 조용히 나빠집니다.**")
        print("      지금 [Add Input] 으로 dogskin-f320 을 붙이고 다시 돌리세요.\n")
    else:
        print("   full 도 없어서 18번 셀에서 멈춥니다 — dogskin-f320 을 붙이세요.\n")

d = crop.switch_tag(df, BEST_CROP)
report = crop.audit(d, cfg=CFG(img_size=IMG_SIZE))

### 🚦 감사 결과 읽기

| 결과 | 의미 | 대응 |
|---|---|---|
| `[1](a)` 정상/병변 배율 **1.5배 이상** | 크롭 배율이 정상/이상을 흘림 | **1단계를 `full` 크롭으로** (자동) |
| `[1](b)` 병변 6종 간 배율 **2배 이상** | 크롭 배율이 병변 종류를 흘림 | 고정 픽셀 크롭 필요 (아래 판단) |
| `[2]` 확대 비율이 50% 넘음 | 없는 디테일을 만들어 냄 | `[1](b)` 와 같은 원인 |
| `[3]` 흐림 비율 30% 넘음 | 원본 사진 품질 한계 | 실사용에서 흐린 사진 거절 (05) |
| `[3]` 정상/병변 선명도 격차 | 배율 차이의 **부작용** | `[1]` 을 고치면 함께 완화됨 |
| `[4]` 라벨 충돌 > 0 | 같은 파일명에 다른 라벨 | 멈추고 알려주세요 |
| `[5]` 이탈량 50px 미만 | 라벨러 오차 | 무시 가능 (크롭이 알아서 잘라 넣음) |
| `[5]` 이탈량 50px 이상 | 좌표 해석 오류 | 멈추고 알려주세요 |

`[1]`과 `[3]`은 **같은 원인**입니다: 박스가 작으면 → 더 확대되고 → 흐려집니다.
따로 고칠 문제가 아닙니다.

### `full` 크롭의 대가 — 병변이 화면 밖으로 나가는 비율

배율 지름길을 막는 가장 확실한 방법은 박스를 아예 안 쓰는 `full`(중앙 정사각) 크롭입니다.
그런데 **병변이 좌우 끝에 있으면 화면에서 잘려 나갑니다.**
그 사진은 "이상"인데 정상처럼 보이니, **1단계 recall의 천장이 데이터 때문에 낮아집니다.**

목표가 recall 0.95인데 천장이 0.90이면 `full`은 못 씁니다. 미리 재고 정합니다.

In [ ]:
loss_full = crop.full_crop_loss(df, "full", cfg=CFG(img_size=IMG_SIZE))

### 얼마나 심각한가 — 사진을 안 보고 맞혀보기 ★

배율 차이가 있다는 건 알았습니다. 그런데 **그게 실제로 얼마나 정답을 흘리는지**는
따로 재야 합니다. 방법은 간단합니다: **픽셀을 한 장도 안 보고** 박스 크기·모양만으로
분류기를 학습시켜 봅니다.

거기서 나오는 점수가 CNN 이 넘어야 하는 **하한선**입니다.

```
CNN macro-F1 0.45  vs  메타데이터만 0.40   →  피부에서 얻은 건 0.05 뿐  🚨
CNN macro-F1 0.45  vs  메타데이터만 0.18   →  대부분 피부에서 얻음     ✅
```

같은 `fold` 를 쓰므로 4번의 CNN 점수와 직접 비교할 수 있습니다.

In [ ]:
# ★ 판단 기준: 크롭 배율로 **이미지에 실제로 보이는** 특징만 씁니다.
floor = crop.shortcut_baseline(d, cfg=CFG(img_size=IMG_SIZE), features="scale_only")

# 참고: 메타데이터 전체를 넣으면 얼마나 나오는지 (종횡비·병변개수·해상도 포함).
# 그것들은 크롭에 안 보이므로 f320 판단에 쓰면 안 됩니다 — 데이터 성질 참고용.
floor_all = crop.shortcut_baseline(d, cfg=CFG(img_size=IMG_SIZE), features="all",
                                   verbose=False)
print(f"\n(참고) 메타데이터 전체 기준: 1단계 AUROC "
      f"{floor_all.get('stage1_auroc_metadata_only', float('nan')):.4f}, "
      f"2단계 macro-F1 {floor_all.get('stage2_macro_f1_metadata_only', float('nan')):.4f}")
print("  이 값이 위보다 높다면, 크롭에 안 보이는 정보(병변 개수 등)가 라벨과")
print("  상관이 있다는 뜻입니다. CNN 은 그걸 못 쓰므로 판단에는 위 숫자를 쓰세요.")

### 판단: 크롭을 다시 만들어야 하나?

위 하한선을 보고 정합니다.

| 하한선 (2단계 macro-F1) | 판단 | 할 일 |
|---|---|---|
| **< 0.30** | 배율 지름길이 약함 | 그냥 진행. 4번에서 CNN 이 하한선을 넘는지 확인 |
| **≥ 0.30** | 배율이 정답을 크게 흘림 | **고정 픽셀 크롭을 만드세요** ↓ |

고정 픽셀 크롭(`f320`)은 병변 중심에서 **항상 320px**을 잘라냅니다.
피부 1mm 가 항상 같은 픽셀 수라 배율로 맞히는 경로가 막힙니다.
대신 큰 병변은 창을 넘어 잘립니다 — 그게 대가입니다.

**로컬 PC 에서** (원본이 있어야 합니다):

```cmd
py prepare_local.py --chunk VL01 --margins -320
py prepare_local.py --finalize
py prepare_local.py --package
```

기존 크롭은 그대로 두고 `f320` 태그만 추가되므로, 올린 뒤 `crop.available_tags()` 에
`f320` 이 보이면 6번의 크롭 비교에 자동으로 포함됩니다.

> 💡 하한선이 애매하면(0.25~0.30) 일단 진행하세요. 4번에서 CNN 점수와 비교한 뒤
> 격차가 작으면 그때 다시 만들면 됩니다. 지금 확실히 아는 건 하한선뿐입니다.

### 눈으로 볼 것 — 딱 하나

아래 표는 클래스마다 한 줄씩입니다. 병변인지 아닌지 판단하지 마세요.
**줄끼리 서로 달라 보이는지만** 보세요.

- 줄마다 달라 보인다 → 모델이 배울 신호가 있습니다 ✅
- 전부 똑같은 털 사진처럼 보인다 → 모델도 구분 못 할 가능성이 큽니다 ⚠️
  (그렇다고 실패는 아닙니다. 모델은 사람이 못 보는 질감 차이를 봅니다.
   다만 기대치를 낮추고, 4번의 macro-F1 을 냉정하게 보셔야 합니다)
- 한 줄 안에서 제각각이다 → 라벨이 섞였을 수 있습니다
- 개 피부/털이 아니라 사람 손·바닥만 보인다 → 크롭이 어긋난 것

In [ ]:
crop.contact_sheet(d, per_class=6)

### 참고: 박스가 어디에 얹혔는지

좌표계가 뒤집혔거나(x↔y) 스케일이 어긋났으면 여기서 드러납니다.
박스가 **크롭 가운데를 크게 차지하는 게 정상**입니다 (margin 1.5 → 폭의 약 2/3).
박스가 구석에 처박혀 있거나 화면을 벗어나면 좌표 해석이 틀린 겁니다.

In [ ]:
crop.preview_with_box(d, n=4)

## 2. 2단계 뷰 만들기

같은 매니페스트에서 **두 개의 뷰**를 만듭니다. 데이터를 복사하는 게 아니라
`label` 컬럼만 다르게 보는 겁니다.

```
df  ──▶ to_stage1()  label: A7 / ABNORMAL      전체 45,885행
    └─▶ to_stage2()  label: A1~A6              병변 23,070행만
```

`fold` / `is_holdout` / `group` 은 그대로 따라옵니다. → 두 단계가 **같은 분할**을 씁니다.

> ⚠️ **두 단계가 서로 다른 크롭을 쓸 수 있습니다.** 위 감사에서 정상/병변의 박스
> 배율이 다르게 나왔다면, 1단계는 `full` 크롭을 씁니다 — ROI 크롭이 배율로
> 정답을 흘리기 때문입니다. 분할은 여전히 공유하므로 누수는 생기지 않습니다.

In [ ]:
# ★ 1단계 크롭 결정 — 규칙은 src/crop.py 에 있습니다 (노트북 셀은 git pull 로 안 바뀜).
#
# ⚠️ **1단계는 BEST_CROP 을 따라오지 않습니다.** 여기서 한 번 헷갈렸습니다.
#    2단계는 병변 형태를 보려고 ROI 크롭을 쓰지만, 1단계에서 ROI 크롭을 쓰면
#    크롭 창 크기가 정답을 흘립니다 (정상 bbox 0.71% vs 병변 1.25%).
#    그 신호는 배포에 없으므로(보호자 사진에는 bbox 가 없음) 검증 점수만 부풀립니다.
choice = crop.choose_stage1_tag(
    best_crop=BEST_CROP,
    scale_gap=report.get("area_ratio_normal_over_lesion", 1.0),
    full_ceiling=loss_full.get("stage1_recall_ceiling", 1.0),
    target_recall=CFG().target_recall_stage1,          # 보통 0.95
    tags=crop.available_tags(),
)
STAGE1_CROP = choice["tag"]
print(f"1단계 크롭: '{STAGE1_CROP}'\n  근거: {choice['why']}")
for w in choice["warnings"]:
    print(w)

# 2단계(병변 종류)는 ROI 크롭을 씁니다 — 형태를 구분하려면 병변을 크게 봐야 합니다.
s1_all = stages.to_stage1(crop.switch_tag(df, STAGE1_CROP))
s2_all = stages.to_stage2(d)

# 두 뷰 각각 누수 재확인 — 뷰를 만드는 과정에서 분할이 깨지지 않았는지
split.verify(s1_all, fold=0, strict=True)
split.verify(s2_all, fold=0, strict=True)

### 학습 전 1분 — 에폭이 몇 분 걸릴지, 왜 그런지

여기서부터 GPU 를 1시간 넘게 씁니다. 그 전에 **처리량을 재두면** 두 가지를 압니다:

1. 에폭당 몇 분인가 → 90분을 쓸지 말지 지금 결정할 수 있습니다
2. **무엇이 병목인가** → 느릴 때 어디를 고칠지

병목은 둘 중 하나입니다. 처방이 정반대라서 추측하면 안 됩니다:

| 병목 | 증상 | 처방 | 하면 안 되는 것 |
|---|---|---|---|
| **입력 파이프라인** (CPU·디스크) | 배치를 키워도 안 빨라짐 | 크롭을 작게 저장, CPU 코어 늘리기 | 배치·해상도 조절 (무효) |
| **GPU** | 배치를 키우면 느려짐 | 모델·해상도 줄이기 | 워커 늘리기 (무효) |

> ⚠️ 이 셀은 **학습이 안 돌고 있을 때** 돌려야 정확합니다.
> 학습 중에 돌리면 GPU·CPU 를 나눠 쓰게 되어 둘 다 틀립니다.

In [ ]:
from src import bench, experiments
from src.config import with_finetune

# ── ① 병목이 어디인가 (CPU 로더 vs GPU) ─────────────────────────
_bcfg = with_finetune(CFG(model_name="resnet50", img_size=IMG_SIZE,
                          balance_strategy="class_weight"), "moderate")
perf = bench.report(split.get_fold(s2_all, 0)[0], _bcfg, classes=CLASSES)

# ── ② 총 예상 시간 ──────────────────────────────────────────────
# ⚠️ 두 단계는 **백본도 데이터 크기도 다릅니다.**
#    1단계 effnetv2_s / 정상 포함 (2배 데이터) / STAGE1_EPOCHS
#    2단계 resnet50   / 병변만              / 25에폭
#    예전에는 둘 다 resnet50·1단계 12에폭으로 계산해서 크게 빗나갔습니다.
_n1 = len(split.get_fold(s1_all, 0)[0])
_n2 = len(split.get_fold(s2_all, 0)[0])
_e1 = experiments.estimate_runtime([STAGE1_MODEL], IMG_SIZE, _n1,
                                   STAGE1_EPOCHS, n_conditions=1)
_e2 = experiments.estimate_runtime(["resnet50"], IMG_SIZE, _n2, 25, n_conditions=1)
_tot = _e1["total_hours"] + _e2["total_hours"]
print(f"\n★ 1단계 {_e1['total_hours']:.1f}시간 + 2단계 {_e2['total_hours']:.1f}시간 "
      f"= 학습 총 **{_tot:.1f}시간**")
print("   (+ 교란 검사·holdout·크롭 확인 별도. 조기 종료가 걸리면 줄어듭니다)")
if _tot > 6:
    print(f"\n🚨 {_tot:.1f}시간은 깁니다. STAGE1_EPOCHS 를 낮추는 걸 고려하세요 "
          f"(03d 에서 best epoch 은 5 였습니다).")

## 3. 1단계 — 정상 / 이상

거의 5:5 라 학습이 수월합니다. 대신 **평가 기준이 다릅니다**:
macro-F1 이 아니라 **재현율(recall)** 이 먼저입니다.

> 오탐(정상인데 병원 가보라고 함) = 보호자가 헛걸음
> 미탐(병변인데 괜찮다고 함) = **놓친 병**
>
> 둘의 비용이 전혀 다르므로 recall 을 0.95로 **먼저 고정**하고,
> 그 조건에서 precision 이 얼마나 나오는지를 봅니다.

📖 [`docs/basics/08_확률보정과_임계값_결정.md`](../docs/basics/08_확률보정과_임계값_결정.md)

In [ ]:
from src.config import with_finetune, with_aug

cfg1 = with_aug(with_finetune(
    CFG(model_name=STAGE1_MODEL, img_size=IMG_SIZE,
        epochs=STAGE1_EPOCHS,
        balance_strategy="none",         # 5:5 라 가중치 불필요
        monitor="macro_f1",
        exp_name=f"stage1_{STAGE1_MODEL}_{STAGE1_CROP}_{IMG_SIZE}"),   # ★ 해상도 포함
    "moderate"), STAGE1_AUG)             # 2단계와 같은 강도로 (조건을 맞춰 비교)

tr1, va1 = split.get_fold(s1_all, cfg1.use_fold)
print(f"1단계  {STAGE1_MODEL} / {STAGE1_AUG}  ·  train {len(tr1):,} / val {len(va1):,}")
print(f"  헤드 lr {cfg1.lr:.1e} / 백본 lr {cfg1.lr * cfg1.backbone_lr_mult:.1e}"
      f" / {cfg1.epochs}에폭 / 배치 {cfg1.resolved_batch_size()}")

m1 = models.build(STAGE1_MODEL, n_classes=len(CLASSES_STAGE1),
                  pretrained=True, drop_rate=cfg1.drop_rate)
dl_tr1, dl_va1, ds_tr1, _ = data.build_loaders(tr1, va1, cfg1, model=m1,
                                               classes=CLASSES_STAGE1)

In [ ]:
# 이미 돌린 게 있으면 이어서 / 건너뜁니다 (세션이 끊겨도 여기서 회복됩니다)
train.print_status(cfg1.exp_name)

res1 = train.fit(m1, dl_tr1, dl_va1, cfg1, ds_train=ds_tr1)
res1.plot()

# ✅ fit 은 끝나면 **best 에폭 가중치**를 m1 에 되돌려 놓습니다.
#    (EMA 를 쓰므로 저장된 best 는 EMA 가중치입니다 — 아래 평가와 일치)

In [ ]:
# 검증셋 점수 → '이상일 확률' → recall 0.95 지점의 임계값
# ★ 결과를 캐시합니다 — 세션이 끊겨 다시 돌릴 때 이 추론을 건너뜁니다.
#    모델·데이터·순서·TTA 중 하나라도 바뀌면 자동으로 다시 계산합니다.
lg1_va, y1_va = train.cached_logits(m1, dl_va1, key="val", exp=cfg1.exp_name,
                                    n_cls=len(CLASSES_STAGE1), device=DEV,
                                    tta_hflip=cfg1.tta_hflip)
scores1 = stages.stage1_scores(lg1_va)
ybin1 = stages.binary_targets(y1_va)

bin1 = evaluate.binary_report(scores1, ybin1, target_recall=cfg1.target_recall_stage1)
THR1 = bin1["threshold"]

### 🚦 1단계 게이트

- **AUROC ≥ 0.80** 이어야 이진 판정이 의미가 있습니다. 0.5는 동전 던지기입니다.
- recall 0.95 조건에서 precision 이 **0.5 미만**이면 오탐이 절반 넘습니다.
  → 쓸 수는 있지만("의심되니 가보세요" 니까) 사용자 신뢰가 빨리 깎입니다. 개선 대상입니다.

In [ ]:
# ★ 판단 기준은 src/gates.py 에 있습니다 — 노트북 셀과 달리 git pull 로 갱신됩니다.
#   STOP(깨진 수준)에서만 멈추고, WANT(기대치) 미달은 경고 후 진행합니다.
from src import gates

gates.stage1(
    auroc=bin1["auroc"],
    threshold=THR1,
    precision=bin1["precision_at_target"],
    floor=floor.get("stage1_auroc_metadata_only"),
    baseline=BASELINE["stage1_auroc"],
    crop_tag=STAGE1_CROP,
    epochs=cfg1.epochs,
)

## 4. 2단계 — 병변 6종

여기가 어려운 쪽입니다. 불균형이 **6.4배**(A2 5,275 ↔ A5 820)이고,
병변 형태끼리 실제로 닮았습니다.

### 파인튜닝 강도를 올려서 갑니다

용어부터. 자주 헷갈리는 지점입니다:

```
전이학습 (transfer learning)   ← 우산 개념
  ├─ linear probe   백본 얼리고 헤드만 학습
  └─ fine-tuning    백본까지 같이 학습     ← 이 리포는 처음부터 이쪽
```

`freeze_backbone()` 은 정의만 되어 있고 **어디서도 호출하지 않습니다.**
그래서 질문은 "파인튜닝을 할까?" 가 아니라 **"얼마나 세게 할까?"** 입니다.

첫 실측(`conservative`, 12에폭)에서 백본 lr 이 `3e-4 × 0.1 = 3e-5` 였는데,
그 결과가 이랬습니다:

| 증상 | 값 | 해석 |
|---|---|---|
| train vs val loss | 1.352 vs 1.474 | 격차 작음 = **과적합 아님** |
| 12에폭 전부 best 갱신 | val loss 한 번도 안 오름 | **수렴 전** |
| macro AUROC vs F1 | 0.8155 vs 0.4865 | 순위는 좋은데 결정이 나쁨 |

**학습 데이터조차 잘 못 맞춥니다.** ImageNet 은 *물체*를 구분하도록 배웠고
우리 과제는 *피부 질감·색의 미세한 차이*라 도메인 격차가 큽니다.
백본이 적응할 시간과 학습률이 필요합니다.

→ 그래서 처음부터 **`moderate`(백본 lr ×0.3) + 25에폭**으로 갑니다.
   비교 기준은 아래 `BASELINE` 에 적어둔 첫 실측값입니다 — 다시 안 돌려도 됩니다.

📖 [`docs/results/STEP4A_베이스라인_실측.md`](../docs/results/STEP4A_베이스라인_실측.md)

In [ ]:
from src.config import with_finetune

cfg2 = with_finetune(
    CFG(model_name="resnet50", img_size=IMG_SIZE,
        epochs=25,                       # 12에폭 전부 best 갱신이었음 → 수렴까지
        balance_strategy="class_weight", # 6.4배 불균형
        monitor="macro_f1",
        exp_name=f"stage2_resnet50_{BEST_CROP}_{IMG_SIZE}"),     # ★ 해상도 포함
    "moderate")                          # 백본 lr ×0.1 → ×0.3

tr2, va2 = split.get_fold(s2_all, cfg2.use_fold)
print(f"2단계  train {len(tr2):,} / val {len(va2):,}")
print(f"  헤드 lr {cfg2.lr:.1e} / 백본 lr {cfg2.lr * cfg2.backbone_lr_mult:.1e}"
      f" / {cfg2.epochs}에폭 / 배치 {cfg2.resolved_batch_size()}")

m2 = models.build("resnet50", n_classes=len(CLASSES),
                  pretrained=True, drop_rate=cfg2.drop_rate)
dl_tr2, dl_va2, ds_tr2, _ = data.build_loaders(tr2, va2, cfg2, model=m2, classes=CLASSES)

In [ ]:
# 증강이 실제로 뭘 하는지 눈으로 보기
import matplotlib.pyplot as plt
from src.data import IMAGENET_MEAN, IMAGENET_STD

x, y = next(iter(dl_tr2))
mean = torch.tensor(IMAGENET_MEAN).view(3,1,1); std = torch.tensor(IMAGENET_STD).view(3,1,1)
fig, axes = plt.subplots(2, 4, figsize=(12, 6.2))
for ax, i in zip(axes.flat, range(min(8, len(x)))):
    ax.imshow((x[i]*std+mean).clamp(0,1).permute(1,2,0)); ax.axis("off")
    ax.set_title(f"{CLASSES[y[i]]} {CLASS_KO[CLASSES[y[i]]][:8]}", fontsize=8)
plt.suptitle("증강 후 실제로 모델이 보는 이미지"); plt.tight_layout(); plt.show()
print("💡 병변이 잘려 나가거나 색이 심하게 변했다면 증강이 너무 센 겁니다.")

In [ ]:
train.print_status(cfg2.exp_name)

res2 = train.fit(m2, dl_tr2, dl_va2, cfg2, ds_train=ds_tr2)
res2.plot()

### 학습 곡선 읽는 법

| 증상 | 의미 | 대응 |
|---|---|---|
| train↓ val↓ 둘 다 계속 하락 | 정상, 더 학습 가능 | epochs 늘리기 |
| train↓ **val↑** | 과적합 시작 | 조기종료 지점, 증강↑ / drop_rate↑ |
| 둘 다 안 내려감 | 학습이 안 됨 | lr 조정, 데이터/라벨 확인 |
| val 이 심하게 출렁임 | 배치가 작거나 lr 이 큼 | batch↑ 또는 lr↓ |

In [ ]:
lg2_va, y2_va = train.cached_logits(m2, dl_va2, key="val", exp=cfg2.exp_name,
                                    n_cls=len(CLASSES), device=DEV,
                                    tta_hflip=cfg2.tta_hflip)
rep2 = evaluate.full_report(lg2_va, y2_va, CLASSES)
rep2.plot_confusion()
rep2.plot_per_class()

### 🚦 2단계 게이트

**macro-F1 이 0.25 미만이면 여기서 멈추고 데이터를 다시 보세요.**
랜덤이 1/6 ≈ 0.167 인데 그것보다 조금 나은 수준이면 파이프라인 어딘가가 깨진 겁니다.

흔한 원인: 라벨 매칭 오류, 크롭 좌표 오류, 클래스 매핑 뒤바뀜.

In [ ]:
from src import gates

cnn_f1 = rep2.metrics["macro_f1"]
a6_recall = rep2.metrics["per_class"]["recall"][CLASSES.index("A6")]

gates.stage2(
    macro_f1=cnn_f1,
    floor=floor.get("stage2_macro_f1_metadata_only"),
    baseline=BASELINE["stage2_macro_f1"],
)

# ── ② 첫 실측 대비: 파인튜닝 강도를 올린 효과 ────────────────────
print(f"\n[첫 실측 대비]  bb×0.1/12ep  →  bb×{cfg2.backbone_lr_mult}/{cfg2.epochs}ep")
print(f"  macro-F1   {BASELINE['stage2_macro_f1']:.4f} → {cnn_f1:.4f}"
      f"   ({cnn_f1 - BASELINE['stage2_macro_f1']:+.4f})")
print(f"  A6 recall  {BASELINE['stage2_a6_recall']:.3f} → {a6_recall:.3f}"
      f"   ({a6_recall - BASELINE['stage2_a6_recall']:+.3f})")

# ── ③ 클래스별 하한선 — 지름길은 평균에 안 보입니다 ──────────────
# ⚠️ 차이의 **부호**가 뜻이 정반대입니다:
#     차이 > 0 인데 작다 → 크기 정보에 얹혀 있을 수 있음 (지름길 의심)
#     차이 < 0           → 크기보다도 못함 = 크기를 **안** 쓰는 중.
#                          지름길이 아니라 그 병변 자체가 어려운 것입니다.
base_rec = floor.get("stage2_recall_metadata_only") or {}
if base_rec:
    cnn_rec = rep2.metrics["per_class"]["recall"]
    print(f"\n  {'클래스':<8}{'하한선':>9}{'CNN':>9}{'차이':>9}   판정")
    leaky, weak = [], []
    for i, c in enumerate(CLASSES):
        b, v = base_rec.get(c, 0.0), cnn_rec[i]
        d = v - b
        if b > 0.3 and 0 <= d < 0.15:
            note = "⚠️ 크기에 얹혀 있을 수 있음"; leaky.append(c)
        elif d < 0:
            note = "→ 크기를 안 씀. 이 병변이 어려운 것"; weak.append(c)
        else:
            note = ""
        print(f"  {c:<8}{b:>9.3f}{v:>9.3f}{d:>+9.3f}   {note}")
    if leaky:
        print(f"\n  ⚠️ 지름길 의심: {', '.join(leaky)} → 6번 배율 교란 검사로 확인")
    if weak:
        print(f"\n  📉 크기보다도 못한 클래스: {', '.join(weak)}")
        print("     지름길 문제가 **아닙니다** — 재크롭으로 해결되지 않습니다.")
        print("     원인은 학습 부족 / 표본 부족 / 병변 난이도입니다.")

# ── ④ 수렴했는가 — 에폭을 더 줘야 하나 ──────────────────────────
h = res2.history
if res2.best_epoch >= len(h) - 2:
    print(f"\n  📈 마지막 에폭({res2.best_epoch})이 최고 — 아직 수렴하지 않았습니다.")
    print(f"     위 cfg2 의 epochs 를 {cfg2.epochs} → {int(cfg2.epochs * 1.6)} 로 올려 다시 돌려보세요.")
else:
    print(f"\n  ✅ epoch {res2.best_epoch} 에서 최고 후 개선 없음 — 수렴했습니다.")
    print("     에폭을 더 늘려도 이 설정으로는 안 오릅니다.")

## 5. 두 단계를 이어붙이기 ★ 여기가 진짜 성능

각 단계를 따로 잘 하는 것과, **이어붙여서** 잘 하는 것은 다릅니다.
**1단계가 놓친 병변은 2단계가 볼 기회조차 없습니다.**
사용자가 실제로 겪는 건 이 이어붙인 결과입니다.

```
사진 → 1단계 ─ '이상 확률' < 임계값 → "정상으로 보입니다"   (2단계는 안 봄)
              └ 임계값 이상 ────────→ 2단계 → "A2 소견이 의심됩니다"
```

⚠️ 평가할 때 중요한 점: **2단계 모델도 정상 사진에 돌려야 합니다.**
실제 서비스에서는 정상 사진도 1단계를 통과하면 2단계로 넘어오니까요.
그래서 두 모델을 **같은 검증셋(정상 포함), 같은 순서**로 돌립니다.

In [ ]:
# 전체 검증셋 = 정상 + 병변. 두 모델을 같은 행·같은 순서로 돌립니다.
va_all = split.get_fold(s1_all, cfg1.use_fold)[1]        # 1단계 뷰 (label_orig 보존)

# ⚠️ 두 단계가 다른 크롭을 쓸 수 있습니다 (1단계 full / 2단계 m1.5).
#    각 모델에는 **그 모델이 학습한 크롭**을 먹여야 합니다.
#    switch_tag 는 image_path 해시로 경로를 다시 계산하므로 행 순서가 보존됩니다.
va_s1 = va_all
va_s2 = crop.switch_tag(va_all, BEST_CROP, verbose=False) if STAGE1_CROP != BEST_CROP \
        else va_all
print(f"1단계 입력 크롭: {STAGE1_CROP}  |  2단계 입력 크롭: {BEST_CROP}")

dl_e1, ds_e1 = data.eval_loader(va_s1, cfg1, model=m1, classes=CLASSES_STAGE1)
dl_e2, ds_e2 = data.eval_loader(va_s2, cfg2, model=m2, classes=CLASSES)

# 순서가 어긋나면 점수가 조용히 엉망이 됩니다 — 반드시 확인
assert len(ds_e1.df) == len(ds_e2.df), \
    f"행 수가 다릅니다: {len(ds_e1.df)} vs {len(ds_e2.df)} — 한쪽 크롭이 빠졌습니다"
assert (ds_e1.df["image_name"].to_numpy() == ds_e2.df["image_name"].to_numpy()).all(), \
    "두 로더의 행 순서가 다릅니다"

# 전체 검증셋(정상 포함) 두 번 — 여기가 노트북에서 가장 무거운 추론입니다. 캐시합니다.
lg1_e, _ = train.cached_logits(m1, dl_e1, key=f"pipe_{STAGE1_CROP}", exp=cfg1.exp_name,
                               n_cls=len(CLASSES_STAGE1), device=DEV, tta_hflip=True)
lg2_e, _ = train.cached_logits(m2, dl_e2, key=f"pipe_{BEST_CROP}", exp=cfg2.exp_name,
                               n_cls=len(CLASSES), device=DEV, tta_hflip=True)

s1_sc = stages.stage1_scores(lg1_e)
y_final = ds_e1.df["label_orig"].to_numpy()               # A1~A7 원래 라벨
print(f"평가 대상 {len(y_final):,}장 (정상 {(y_final == NORMAL_LABEL).sum():,} / "
      f"병변 {(y_final != NORMAL_LABEL).sum():,})")

In [ ]:
pipe = stages.pipeline_report(s1_sc, lg2_e, y_final, threshold=THR1)
stages.plot_pipeline_confusion(pipe)

## 6. 실사용 견고성 검사 ★ 여기서 진짜가 드러납니다

지금까지의 점수는 모두 **우리가 만든 크롭** 위에서 잰 것입니다.
그 크롭은 병변을 정중앙에 두고, 병변 크기에 맞춰 배율을 정했습니다.
보호자 사진은 둘 다 아닙니다.

그래서 검증셋을 일부러 그렇게 망가뜨려 보고 점수 하락폭을 잽니다.
**하락폭이 곧 실사용 위험도**입니다. 학습은 안 하니 몇 분이면 됩니다.

| 하락폭 | 판정 |
|---|---|
| 15% 미만 | ✅ 견고 |
| 15~30% | ⚠️ 상당히 의존 — 개선 여지 큼 |
| 30% 이상 | 🚨 실사용에서 무너짐 |

> 💡 **`f320`의 효과는 하한선이 아니라 이 숫자로 판정하세요.**
> 하한선(`shortcut_baseline`)은 `bbox` 컬럼을 쓰기 때문에 크롭 방식을 바꿔도
> 거의 안 변합니다. "데이터에 상관이 있나"와 "모델이 그걸 썼나"는 다른 질문입니다.

In [ ]:
from src import robust

rb = robust.report(m2, va2, cfg2, CLASSES, n=2000)

### 임계값을 바꾸면 무엇이 바뀌나

임계값 하나가 이 시스템의 **성격**을 정합니다.
낮추면 놓치는 병변은 줄고 헛알림이 늘어납니다.

이 프로젝트의 목적("의심된다까지 알려주기")에서는 **놓치지 않는 쪽**이 맞습니다.
다만 헛알림이 너무 많으면 보호자가 알림을 무시하게 되므로, 표를 보고 균형점을 잡으세요.

In [ ]:
import pandas as pd
import numpy as np

rows = []
for t in np.quantile(s1_sc, [0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65]):
    r = stages.pipeline_report(s1_sc, lg2_e, y_final, threshold=float(t), show=False)
    rows.append({"임계값": round(float(t), 4),
                 "놓친병변": r["lesion_missed"],
                 "스크리닝recall": round(r["lesion_screening_recall"], 4),
                 "헛알림비율": round(r["false_alarm_rate"], 4),
                 "종류정확도": round(r["kind_accuracy_given_routed"], 4),
                 "최종macroF1": round(r["final_macro_f1"], 4)})
tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))
print(f"\n선택한 임계값: {THR1:.4f} (recall {cfg1.target_recall_stage1:.0%} 목표 기준)")
print("💡 '스크리닝recall' 이 0.95를 넘는 가장 큰 임계값을 고르면 헛알림이 최소가 됩니다.")

## 7. 결과 저장 — 여기까지가 필수입니다

노트북 04·05 가 읽어갈 선택 결과를 남깁니다.
아래 8번(해상도 실험)이 끝나면 이 셀을 다시 돌려 최종본으로 갱신하세요.

In [ ]:
import json

W = env.work_root()
W.mkdir(parents=True, exist_ok=True)

(W/"best_crop.txt").write_text(BEST_CROP)
(W/"stage1_threshold.json").write_text(json.dumps({
    "threshold": THR1,
    "target_recall": cfg1.target_recall_stage1,
    "auroc": bin1["auroc"],
    "precision_at_target": bin1["precision_at_target"],
    "stage1_crop": STAGE1_CROP,        # ← 1단계는 다른 크롭일 수 있습니다
    "stage2_crop": BEST_CROP,
    # ★ 실험 이름을 남깁니다. with_finetune 이 프리셋 이름을 붙이므로
    #   (stage1_resnet50_full → stage1_resnet50_full_moderate) 노트북 05 가
    #   이름을 추측하면 못 찾습니다.
    "stage1_exp": cfg1.exp_name,
    "stage2_exp": cfg2.exp_name,
    "img_size": IMG_SIZE,
    "finetune": {"backbone_lr_mult": cfg2.backbone_lr_mult, "epochs": cfg2.epochs},
    "audit": {k: v for k, v in report.items() if not isinstance(v, (dict, list))},
}, indent=2, ensure_ascii=False))

summary = {
    "stage1": {"crop": STAGE1_CROP, "auroc": bin1["auroc"], "threshold": THR1,
               "model": STAGE1_MODEL, "aug": STAGE1_AUG,
               "epochs": cfg1.epochs, "backbone_lr_mult": cfg1.backbone_lr_mult},
    "stage2": {"crop": BEST_CROP, "macro_f1": rep2.metrics["macro_f1"],
               "ci": list(rep2.ci[1:]),
               "per_class_recall": dict(zip(CLASSES, rep2.metrics["per_class"]["recall"])),
               "epochs": cfg2.epochs, "backbone_lr_mult": cfg2.backbone_lr_mult},
    "floor": {"stage1_auroc": floor.get("stage1_auroc_metadata_only"),
              "stage2_macro_f1": floor.get("stage2_macro_f1_metadata_only")},
    "pipeline": {k: v for k, v in pipe.items() if not isinstance(v, (dict, list))},
    "robustness": {"scale_rel_drop": rb["scale"].get("_summary", {}).get("rel_drop"),
                   "shift_rel_drop": rb["shift"].get("_summary", {}).get("rel_drop")},
    "baseline_first_run": BASELINE,
    "exp_names": {"stage1": cfg1.exp_name, "stage2": cfg2.exp_name},
    "img_size": IMG_SIZE,
}
(W/"reports").mkdir(parents=True, exist_ok=True)
(W/"reports"/"step4a_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False))

# ★ 다음 노트북(04/05)에 넘길 것만 한 폴더로 모읍니다.
#   work/ 안쪽에 두면 Kaggle 이 데이터셋으로 만들 때 빠집니다 (실제로 빠졌습니다).
train.export_release(
    exps=[cfg1.exp_name, cfg2.exp_name],
    meta={"1단계 크롭": STAGE1_CROP, "2단계 크롭": BEST_CROP, "입력": f"{IMG_SIZE}px",
          "1단계 AUROC": f"{bin1['auroc']:.4f}", "임계값": f"{THR1:.4f}",
          "2단계 macro-F1": f"{rep2.metrics['macro_f1']:.4f}",
          "배율 하락": f"{rb['scale'].get('_summary', {}).get('rel_drop', float('nan')):.1%}"},
    files={"stage1_threshold.json": json.loads((W/"stage1_threshold.json").read_text()),
           "reports/step4a_summary.json": summary},
)

print("저장 완료")
print(f"  1단계 크롭 {STAGE1_CROP} / 2단계 크롭 {BEST_CROP}")
print(f"  work_root: {W}")
print("\n" + "=" * 60)
print(" STEP 4A 결과 (이 블록을 복사해서 공유하세요)")
print("=" * 60)
print(f"  1단계 AUROC        {bin1['auroc']:.4f}   (첫 실측 {BASELINE['stage1_auroc']:.4f})")
print(f"  2단계 macro-F1     {rep2.metrics['macro_f1']:.4f}   (첫 실측 {BASELINE['stage2_macro_f1']:.4f})")
print(f"  A6 recall          {rep2.metrics['per_class']['recall'][CLASSES.index('A6')]:.3f}"
      f"   (첫 실측 {BASELINE['stage2_a6_recall']:.3f})")
print(f"  스크리닝 recall    {pipe['lesion_screening_recall']:.4f}")
print(f"  헛알림 비율        {pipe['false_alarm_rate']:.4f}")
print(f"  배율 교란 하락     {rb['scale'].get('_summary', {}).get('rel_drop', float('nan')):.1%}")
print(f"  위치 교란 하락     {rb['shift'].get('_summary', {}).get('rel_drop', float('nan')):.1%}")
print("=" * 60)

---
## ✅ 정리

| 확인한 것 | 어디서 | 통과 기준 |
|---|---|---|
| 크롭이 배율로 정답을 흘리지 않는가 | 1번 감사 | 정상/병변 1.5배 미만 |
| `full` 로 바꿔도 병변을 안 잃는가 | 1번 `full_crop_loss` | 천장 ≥ 0.95 |
| 사진 없이 얼마나 맞히는가 | 1번 `shortcut_baseline` | 2단계 < 0.30 |
| 1단계가 정상/이상을 구분한다 | 3번 | AUROC > 0.80 |
| 2단계가 하한선을 넘는다 | 4번 | 차이 > 0.15 |
| **이어붙인 실제 성능** | 5번 | 스크리닝 recall ≥ 0.95 |
| **실사용에서 버티는가** | 6번 | 배율·위치 하락 < 15% |

## 다음 단계

**`03b_증강_배율강건성.ipynb`** — 2단계에 남은 배율 하락 20.4% 를 확대 증강으로 잡습니다.

384 에서 최악 조건이 **축소(0.5x) → 확대(2x)** 로 바뀌었습니다. 축소는 픽셀이 사라지는
물리 문제라 해상도로만 풀리지만, 확대는 훈련 분포 문제라 **증강으로 풀 수 있습니다.**

⚠️ **`04`(백본 비교)는 배율 하락이 잡히기 전까지 보류합니다.**
무너지는 기준 위에서 6개 모델을 비교하면 "배율을 가장 잘 읽는 모델" 을 뽑게 됩니다.
그건 실사용에서 가장 먼저 무너지는 모델입니다.

📖 [`docs/basics/09_ViT와_최신_백본_지도_2026.md`](../docs/basics/09_ViT와_최신_백본_지도_2026.md) ·
[`docs/cautions/08_2단계_파이프라인_설계_주의점.md`](../docs/cautions/08_2단계_파이프라인_설계_주의점.md) ·
[`docs/results/STEP4A_베이스라인_실측.md`](../docs/results/STEP4A_베이스라인_실측.md)
